In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

# Neo4j Driver
import neo4j

# Always close the driver when done
neo4j_driver = neo4j.GraphDatabase.driver(
    NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

try:
    neo4j_driver.verify_connectivity()
    print("✅ Connected to Neo4j successfully")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    raise  # stop execution if DB is unreachable

# neo4j_driver.close()

# LLM and Embedding Model
# from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.llm import OllamaLLM
from neo4j_graphrag.embeddings import SentenceTransformerEmbeddings

# os.environ["OPENAI_API_KEY"] = os.getenv("Groq_API_Key")

# llm = OpenAILLM(
#     model_name="llama-3.3-70b-versatile",  # or any Groq model you prefer
#     model_params={
#         # "response_format": {"type": "json_object"},
#         "temperature": 0
#     },
#     base_url="https://api.groq.com/openai/v1",  # point to Groq's endpoint
#     api_key=os.getenv("Groq_API_Key")           # your Groq API key
# )

# LLM - Ollama (no API key needed)
llm = OllamaLLM(
    model_name="qwen2.5:1.5b",
    model_params={
        "options": {"temperature": 0},
        "format": "json"
    }
)

# Core Academic Entities
academic_nodes = [
    "Department",
    "Program",
    "Course",
    "Semester",
    "Faculty"
]

# Fee System
fee_nodes = [
    "FeeStructure",
    "FeeComponent",
    "FeeType",
    "StudyMode"
]

# Admission & Metadata
admission_nodes = [
    "Eligibility",
    "Requirement",
    "Duration",
    "CreditInfo"
]

# Optional (for GraphRAG context chunks)
rag_nodes = [
    "Chunk"
]

node_labels = academic_nodes + fee_nodes + admission_nodes + rag_nodes

rel_types = [

    # Academic Structure
    "OFFERS",              # Department → Program
    "HAS_SEMESTER",        # Program → Semester
    "HAS_COURSE",          # Semester → Course
    "HAS_PREREQUISITE",    # Course → Course
    "TAUGHT_BY",           # Course → Faculty

    # Program Metadata
    "HAS_ELIGIBILITY",     # Program → Eligibility
    "HAS_DURATION",        # Program → Duration
    "HAS_CREDIT_INFO",     # Program → CreditInfo

    # Fee Structure
    "HAS_FEE_STRUCTURE",   # Program → FeeStructure
    "INCLUDES_FEE",        # FeeStructure → FeeComponent
    "OF_TYPE",             # FeeComponent → FeeType
    "APPLIES_TO",          # FeeComponent → StudyMode

    # GraphRAG Linking
    "FROM_CHUNK"           # Entity → Chunk
]

patterns = [
    ("Department", "OFFERS", "Program"),
    ("Program", "HAS_SEMESTER", "Semester"),
    ("Semester", "HAS_COURSE", "Course"),
    ("Course", "HAS_PREREQUISITE", "Course"),
    ("Course", "TAUGHT_BY", "Faculty"),

    ("Program", "HAS_FEE_STRUCTURE", "FeeStructure"),
    ("FeeStructure", "INCLUDES_FEE", "FeeComponent"),
    ("FeeComponent", "OF_TYPE", "FeeType"),
    ("FeeComponent", "APPLIES_TO", "StudyMode"),

    ("Program", "HAS_ELIGIBILITY", "Eligibility")
]



#create text embedder
embedder = SentenceTransformerEmbeddings(
    model="all-MiniLM-L6-v2"  # free, lightweight, runs locally
)

# define prompt template
prompt_template = '''
You are an AI system designed to extract structured academic and admission-related knowledge 
from university documents and convert it into a Knowledge Graph.

CRITICAL: Return ONLY a raw JSON object. 
- NO markdown code blocks
- NO ```json or ``` fences
- NO preamble or explanation
- Start your response directly with {{ and end with }}

Your task is to:
1. Identify entities (nodes) from the text.
2. Assign each entity a valid type (label).
3. Extract relationships between entities.
4. Maintain correct direction of relationships.

### STRICT INSTRUCTIONS:
- ONLY use the provided node labels and relationship types.
- DO NOT invent new labels or relationships.
- DO NOT include unnecessary or duplicate entities.
- Normalize entity names (e.g., "BS Computer Science" should be consistent).
- Each node MUST have a unique string ID.
- Relationships MUST correctly reference node IDs.

### DOMAIN UNDERSTANDING:
The text may include:
- Programs (e.g., BS Computer Science)
- Departments (e.g., Computer Science Department)
- Courses and semesters
- Fee structures (tuition, admission fee, etc.)
- Study modes (Regular, Self Support, Weekend)
- Eligibility criteria

### NORMALIZATION RULES (STRICTLY FOLLOW):
- "BS Computer Science", "B.S. Computer Science", "BSc Computer Science", 
  "Bachelor of Science in Computer Science" → ALWAYS use: "BS Computer Science"
- "MS Computer Science", "M.S. Computer Science" → ALWAYS use: "MS Computer Science"  
- "BS Information Technology" variants → ALWAYS use: "BS Information Technology"
- "BS Software Engineering" variants → ALWAYS use: "BS Software Engineering"
- "BS Artificial Intelligence" variants → ALWAYS use: "BS Artificial Intelligence"
- "MS Artificial Intelligence" variants → ALWAYS use: "MS Artificial Intelligence"

### ALLOWED SCHEMA:
{schema}

### EXAMPLE:
Input: "BS Computer Science program has a tuition fee of 35150 per semester."

Output:
{{
  "nodes": [
    {{"id": "0", "label": "Program", "properties": {{"name": "BS Computer Science"}}}},
    {{"id": "1", "label": "FeeComponent", "properties": {{"amount": "35150"}}}},
    {{"id": "2", "label": "FeeType", "properties": {{"name": "Tuition Fee"}}}}
  ],
  "relationships": [
    {{"type": "INCLUDES_FEE", "start_node_id": "0", "end_node_id": "1", "properties": {{}}}},
    {{"type": "OF_TYPE", "start_node_id": "1", "end_node_id": "2", "properties": {{}}}}
  ]
}}

### INPUT TEXT:
{text}
'''

✅ Connected to Neo4j successfully


d:\Conda\envs\my_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3721.91it/s]


In [3]:
# Knowledge Graph Builder
from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import FixedSizeSplitter
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

kg_builder_pdf = SimpleKGPipeline(
   llm=llm,
   driver=neo4j_driver,
   text_splitter=FixedSizeSplitter(chunk_size=800, chunk_overlap=150),
   embedder=embedder,
   entities=node_labels,
   relations=rel_types,
   prompt_template=prompt_template,
   from_pdf=True
)

pdf_file_paths = [
            r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\CS-Prospectus-2025.pdf',
            r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\CS_Fee_Structure.pdf',
            r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\CS_Eligibility.pdf',
            r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Sem Regulations.pdf'
         ]

for path in pdf_file_paths:
    try:
        result = await kg_builder_pdf.run_async(file_path=path)
        print(f"✓ {path}: {result}")
    except Exception as e:
        print(f"✗ {path} failed: {e}")

C:\Users\asmaa\AppData\Local\Temp\ipykernel_18276\3347925245.py:5: DeprecationWarning: `from_pdf` is deprecated and will be removed in a future version; use `from_file` instead.
  kg_builder_pdf = SimpleKGPipeline(


✓ W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\CS-Prospectus-2025.pdf: run_id='ddb83077-ce7d-4647-98d2-79742cf04818' result={'resolver': {'number_of_nodes_to_resolve': 249, 'number_of_created_nodes': 176}}
✓ W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\CS_Fee_Structure.pdf: run_id='98f4901e-2f19-4968-ba49-57ce38844f41' result={'resolver': {'number_of_nodes_to_resolve': 206, 'number_of_created_nodes': 187}}
✓ W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\CS_Eligibility.pdf: run_id='cfb072de-d0a9-4fbc-a496-843ab1a31d6a' result={'resolver': {'number_of_nodes_to_resolve': 223, 'number_of_created_nodes': 204}}


LLM response has improper format for chunk_index=35
LLM response has improper format for chunk_index=65


✓ W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Sem Regulations.pdf: run_id='2af2fdca-1eb7-4848-b772-e970a92df7de' result={'resolver': {'number_of_nodes_to_resolve': 433, 'number_of_created_nodes': 340}}


In [3]:
import os
import pandas as pd
from pathlib import Path
from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import FixedSizeSplitter
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

# ── 1. File Converters ────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    """Remove noise that confuses small models."""
    import re
    # Remove excessive whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove page numbers like "Page 1 of 10" or just "1"
    text = re.sub(r'\bPage\s+\d+\s+of\s+\d+\b', '', text, flags=re.IGNORECASE)
    # Remove repeated dashes/underscores (table borders)
    text = re.sub(r'[-_]{3,}', '', text)
    # Remove excessive spaces
    text = re.sub(r' {3,}', ' ', text)
    return text.strip()


def pdf_to_text(file_path: str) -> str:
    """Extract clean text from PDF."""
    import pypdf
    text = ""
    with open(file_path, 'rb') as f:
        reader = pypdf.PdfReader(f)
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    return clean_text(text)


def csv_to_text(file_path: str) -> str:
    """Convert CSV to readable sentence-style text."""
    df = pd.read_csv(file_path)
    lines = []
    
    # Convert each row to a readable sentence
    for _, row in df.iterrows():
        row_text = ", ".join(
            f"{col}: {val}" 
            for col, val in row.items() 
            if pd.notna(val)
        )
        lines.append(row_text)
    
    return clean_text("\n".join(lines))


def txt_to_text(file_path: str) -> str:
    """Read and clean plain text file."""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        return clean_text(f.read())


def file_to_text(file_path: str) -> str:
    """Route file to correct converter based on extension."""
    ext = Path(file_path).suffix.lower()
    
    if ext == '.pdf':
        return pdf_to_text(file_path)
    elif ext == '.csv':
        return csv_to_text(file_path)
    elif ext in ['.txt', '.md']:
        return txt_to_text(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")


# ── 2. Two Pipelines (PDF native + Text) ─────────────────────────────────────

# For PDFs using native parser
kg_builder_pdf = SimpleKGPipeline(
    llm=llm,
    driver=neo4j_driver,
    text_splitter=FixedSizeSplitter(chunk_size=1200, chunk_overlap=200),
    embedder=embedder,
    entities=node_labels,
    relations=rel_types,
    prompt_template=prompt_template,
    from_pdf=True
)

# For pre-converted text (CSV, TXT, or manually cleaned PDFs)
kg_builder_text = SimpleKGPipeline(
    llm=llm,
    driver=neo4j_driver,
    text_splitter=FixedSizeSplitter(chunk_size=1200, chunk_overlap=200),
    embedder=embedder,
    entities=node_labels,
    relations=rel_types,
    prompt_template=prompt_template,
    from_pdf=False
)


# ── 3. Your File List (mixed types) ──────────────────────────────────────────

file_paths = [
    r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Documents\CS-Prospectus-2025.pdf',
    r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Documents\CS_Fee_Structure.pdf',
    r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Documents\CS_Eligibility.pdf',
    r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Documents\Sem Regulations.pdf',
    r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Documents\Curriculum.csv',
    r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Documents\Faculty.csv',
    r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Documents\Fee_Structure.csv',
    r'W:\7th Sem\FYP\Project\AI-INBOUND-CALLING-AGENT\Asmaad\Documents\Programs.csv'
]


# ── 4. Smart Runner ───────────────────────────────────────────────────────────

async def process_files(file_paths: list):
    """Process any mix of PDF, CSV, TXT files."""
    
    for path in file_paths:
        ext = Path(path).suffix.lower()
        print(f"\n📄 Processing [{ext.upper()}]: {Path(path).name}")
        
        try:
            if ext == '.pdf':
                # Use native PDF pipeline directly
                result = await kg_builder_pdf.run_async(file_path=path)
                
            elif ext in ['.csv', '.txt', '.md']:
                # Convert to clean text first, then run
                print(f"   → Converting to clean text...")
                text = file_to_text(path)
                print(f"   → Text length: {len(text)} characters")
                result = await kg_builder_text.run_async(text=text)
                
            else:
                print(f"   ⚠️ Skipping unsupported file type: {ext}")
                continue
                
            print(f"   ✓ Done: {result}")
            
        except Exception as e:
            print(f"   ✗ Failed: {e}")


# ── 5. Run ────────────────────────────────────────────────────────────────────

await process_files(file_paths)

C:\Users\asmaa\AppData\Local\Temp\ipykernel_7856\1088321471.py:76: DeprecationWarning: `from_pdf` is deprecated and will be removed in a future version; use `from_file` instead.
  kg_builder_pdf = SimpleKGPipeline(
C:\Users\asmaa\AppData\Local\Temp\ipykernel_7856\1088321471.py:88: DeprecationWarning: `from_pdf` is deprecated and will be removed in a future version; use `from_file` instead.
  kg_builder_text = SimpleKGPipeline(



📄 Processing [.PDF]: CS-Prospectus-2025.pdf
   ✓ Done: run_id='b99ef7fd-88ee-4e04-975d-8eef1967238d' result={'resolver': {'number_of_nodes_to_resolve': 443, 'number_of_created_nodes': 369}}

📄 Processing [.PDF]: CS_Fee_Structure.pdf


LLM response has improper format for chunk_index=3


   ✓ Done: run_id='335f8ba2-2741-4c00-892e-b98fafe896ea' result={'resolver': {'number_of_nodes_to_resolve': 415, 'number_of_created_nodes': 391}}

📄 Processing [.PDF]: CS_Eligibility.pdf


LLM response has improper format for chunk_index=0


   ✓ Done: run_id='2e204d9a-5bcb-42c8-a522-d0b2909376b4' result={'resolver': {'number_of_nodes_to_resolve': 418, 'number_of_created_nodes': 391}}

📄 Processing [.PDF]: Sem Regulations.pdf


LLM response has improper format for chunk_index=21


   ✓ Done: run_id='696c4265-79a1-451a-94f9-887361900ec5' result={'resolver': {'number_of_nodes_to_resolve': 567, 'number_of_created_nodes': 451}}

📄 Processing [.CSV]: Curriculum.csv
   → Converting to clean text...
   → Text length: 8314 characters
   ✓ Done: run_id='85e95bf3-915a-4e06-86f6-a8257b0c1d77' result={'resolver': {'number_of_nodes_to_resolve': 519, 'number_of_created_nodes': 466}}

📄 Processing [.CSV]: Faculty.csv
   → Converting to clean text...
   → Text length: 688 characters
   ✓ Done: run_id='ae5d1f03-dec7-4be5-853d-583689433fa1' result={'resolver': {'number_of_nodes_to_resolve': 487, 'number_of_created_nodes': 466}}

📄 Processing [.CSV]: Fee_Structure.csv
   → Converting to clean text...
   → Text length: 2263 characters
   ✓ Done: run_id='d6c28c32-8e32-4b05-b158-e37e0abda256' result={'resolver': {'number_of_nodes_to_resolve': 496, 'number_of_created_nodes': 469}}

📄 Processing [.CSV]: Programs.csv
   → Converting to clean text...
   → Text length: 3062 characters
   

In [4]:
# Run this in a new cell to merge all variations into one node
with neo4j_driver.session() as session:
    
    # Merge all BS CS variations into one canonical node
    session.run("""
        MATCH (p:Program)
        WHERE p.name IN [
            'B.S. Computer Science',
            'BS Computer Science Program', 
            'Bachelor of Science in Computer Science',
            "Bachelor's Science Computer Science",
            'BSc Computer Science',
            'Bachelor of Science (BSc) in Computer Science',
            '4-year BS program',
            "Bachelor's Degree in Computer Science"
        ]
        WITH p
        MATCH (canonical:Program {name: 'BS Computer Science'})
        // Move all relationships to canonical node
        CALL apoc.refactor.mergeNodes([canonical, p], {
            properties: 'discard',
            mergeRels: true
        })
        YIELD node
        RETURN node.name
    """)
    print("✅ Nodes merged")

✅ Nodes merged


In [5]:
import psutil
print(f"Available RAM: {psutil.virtual_memory().available / 1024**3:.1f} GB")

Available RAM: 1.2 GB


In [6]:
from neo4j_graphrag.indexes import create_vector_index
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG, RagTemplate
from neo4j_graphrag.llm import OllamaLLM

In [7]:
# ── 1. Vector Index ───────────────────────────────────────────────────────────
try:
    create_vector_index(
        neo4j_driver,                    # ← was: driver
        name="text_embeddings",
        label="Chunk",
        embedding_property="embedding",
        dimensions=384,                  # ← 384 for SentenceTransformer
        similarity_fn="cosine"
    )
    print("✅ Vector index created")
except Exception as e:
    if "already exists" in str(e).lower():
        print("ℹ️ Index already exists, skipping")
    else:
        raise

✅ Vector index created


In [8]:
# ── 2. Vector Retriever ───────────────────────────────────────────────────────
vector_retriever = VectorRetriever(
    neo4j_driver,                       
    index_name="text_embeddings",
    embedder=embedder,
    return_properties=["text"],
)


In [9]:
# ── 3. Graph Retriever ────────────────────────────────────────────────────────
graph_retriever = VectorCypherRetriever(
    neo4j_driver,                        
    index_name="text_embeddings",
    embedder=embedder,
    retrieval_query="""
WITH node AS chunk

// Fuzzy match - catches name variations
MATCH (chunk)<-[:FROM_CHUNK]-(p:Program)

OPTIONAL MATCH (p)<-[:OFFERS]-(d:Department)
OPTIONAL MATCH (p)-[:HAS_SEMESTER]->(s:Semester)
OPTIONAL MATCH (s)-[:HAS_COURSE]->(c:Course)
OPTIONAL MATCH (p)-[:HAS_FEE_STRUCTURE]->(fs:FeeStructure)
OPTIONAL MATCH (fs)-[:INCLUDES_FEE]->(fc:FeeComponent)
OPTIONAL MATCH (fc)-[:OF_TYPE]->(ft:FeeType)
OPTIONAL MATCH (fc)-[:APPLIES_TO]->(sm:StudyMode)

WITH chunk, p, d, s, c, fs, fc, ft, sm

RETURN 
chunk.text + '\n' +
'Program: '       + coalesce(p.name,'')  + '\n' +
'Department: '    + coalesce(d.name,'')  + '\n' +
'Semester: '      + coalesce(s.name,'')  + '\n' +
'Course: '        + coalesce(c.name,'')  + '\n' +
'Fee Component: ' + coalesce(fc.name,'') + '\n' +
'Fee Type: '      + coalesce(ft.name,'') + '\n' +
'Study Mode: '    + coalesce(sm.name,'') AS info
"""
)

In [10]:
# ── 4. LLM for RAG ───────────────────────────────────────────────────────────
rag_llm = OllamaLLM(                     
    model_name="qwen2.5:1.5b",            
    model_params={
        "options": {"temperature": 0}
        # no "format": "json" here — RAG needs natural language output
    }
)


# Separate models for KG building vs RAG answering
# kg_llm = OllamaLLM(
#     model_name="llama3.2:8b",       # small, fast, good enough for extraction
#     model_params={
#         "options": {"temperature": 0},
#         "format": "json"
#     }
# )

In [11]:
# ── 5. RAG Template ───────────────────────────────────────────────────────────
rag_template = RagTemplate(
    template='''
You are a university academic assistant. Answer ONLY what is asked.
Be concise and direct. List only relevant information.
Do NOT mention course codes unless specifically asked.
Do NOT explain what you think the user is looking for.

If exact data is missing, say "Not found in context".

# Question:
{query_text}

# Context:
{context}

# Answer (be direct and concise):
''',
    expected_inputs=['query_text', 'context']
)


In [12]:
# ── 6. GraphRAG instances ─────────────────────────────────────────────────────
vector_rag = GraphRAG(
    llm=rag_llm,
    retriever=vector_retriever,
    prompt_template=rag_template
)

graph_rag = GraphRAG(
    llm=rag_llm,
    retriever=graph_retriever,
    prompt_template=rag_template
)

In [13]:
# ── 7. Query ──────────────────────────────────────────────────────────────────
q = "What programs are offered by the Computer Science Department?"

print("=== VECTOR RAG ===")
print(vector_rag.search(q, retriever_config={'top_k': 10}).answer)

print("\n=== GRAPH RAG ===")
print(graph_rag.search(q, retriever_config={'top_k': 10}).answer)

=== VECTOR RAG ===
The Computer Science Department offers the following programs:

1. BS in Computer Science

2. MS Computer Science

3. PhD Computer Science

=== GRAPH RAG ===
The provided text contains a list of course codes, fee components, and study modes for various semesters. The courses cover topics such as mathematics, computer science, database systems, digital logic design, multivariable calculus, operating systems, data mining, artificial intelligence, software engineering, probability & statistics, and more. Each entry includes the course code (e.g., MATH-5101), fee component (e.g., 4(3-1)), and study mode (e.g., One Semester).


In [14]:
# Run this in a new cell to inspect your graph
from neo4j_graphrag.indexes import create_vector_index

with neo4j_driver.session() as session:
    # Check how many nodes were created
    result = session.run("MATCH (n) RETURN labels(n) AS label, count(n) AS count")
    for record in result:
        print(record["label"], "→", record["count"])

    # Check if FROM_CHUNK relationships exist
    result = session.run("MATCH ()-[:FROM_CHUNK]->() RETURN count(*) AS count")
    print("FROM_CHUNK relationships:", result.single()["count"])

    # Check programs linked to chunks
    result = session.run("""
        MATCH (c:Chunk)<-[:FROM_CHUNK]-(p:Program) 
        RETURN p.name, count(c) AS chunks
    """)
    for record in result:
        print(record["p.name"], "→", record["chunks"], "chunks")

['__KGBuilder__', 'Document'] → 7
['__KGBuilder__', 'Chunk'] → 152
['__KGBuilder__', 'Department', '__Entity__'] → 15
['__KGBuilder__', '__Entity__', 'Program'] → 46
['__KGBuilder__', '__Entity__', 'Course'] → 154
['__KGBuilder__', '__Entity__', 'Eligibility'] → 42
['__KGBuilder__', '__Entity__', 'Semester'] → 39
['__KGBuilder__', '__Entity__', 'StudyMode'] → 2
['__KGBuilder__', '__Entity__', 'Duration'] → 14
['__KGBuilder__', '__Entity__', 'FeeStructure'] → 4
['__KGBuilder__', '__Entity__', 'FeeComponent'] → 6
['__KGBuilder__', '__Entity__', 'FeeType'] → 3
['__KGBuilder__', '__Entity__', 'Faculty'] → 3
['__KGBuilder__', '__Entity__', 'Requirement'] → 26
['__KGBuilder__', '__Entity__', 'CreditInfo'] → 2
FROM_CHUNK relationships: 555
BS Computer Science → 60 chunks
Computer Science → 2 chunks
BS Urdu - 4-year Program → 1 chunks
BS Applied Physics → 1 chunks
Expository Writing → 1 chunks
Islamic Studies → 1 chunks
Translation of Holy Quran-II → 2 chunks
Data Science → 2 chunks
BS in Data